# Cleaning and Validating Raw Data

## Step 1: Load And Inspect The Raw Data

In this step, we load the raw harvest file and do a quick structural review: shape, sample rows, missing values, and initial data types. The goal is to understand the dataset before making any changes.


In [198]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/palma_raw.csv")

print("Rows and columns:", df.shape)

print("\nQuick look at the data:")
display(df.head(10))

print("\nMissing values:")
print(df.isna().sum())

print("\nData Types:")
print(df.dtypes)

Rows and columns: (447, 7)

Quick look at the data:


,date,24,8,2,Total,Weight,Average
0,12/4/15,NaN,NaN,NaN,622.0,1320.0,2.122187
1,12/19/15,NaN,NaN,NaN,570.0,1470.0,2.578947
2,12/30/15,NaN,NaN,NaN,420.0,1190.0,2.833333
3,1/5/16,NaN,NaN,NaN,680.0,1790.0,2.632353
4,1/12/16,NaN,NaN,NaN,735.0,1920.0,2.612245
5,1/20/16,NaN,NaN,NaN,937.0,2410.0,2.572038
6,1/26/16,722.0,329.0,67.0,1118.0,2820.0,2.522361
7,2/2/16,988.0,615.0,0.0,1603.0,4030.0,2.514036
8,2/9/16,1047.0,0.0,90.0,1137.0,2480.0,2.181179
9,2/10/16,0.0,517.0,0.0,517.0,1390.0,2.688588



Missing values:
date        0
24         27
8          29
2          29
Total      18
Weight     26
Average    44
dtype: int64

Data Types:
date        object
24         float64
8          float64
2          float64
Total      float64
Weight     float64
Average    float64
dtype: object


## Step 2: Standardize Column Names

We rename the original columns so they are descriptive, consistent, and easier to use in later analysis. This also avoids awkward numeric column names like `24`, `8`, and `2`.


In [199]:
df = df.rename(columns = {
    "date": "harvest_date",
    "24": "bunches_24ha",
    "8": "bunches_8ha",
    "2": "bunches_2ha",
    "Total": "total_bunches",
    "Weight": "total_weight_kg",
    "Average": "avg_bunch_weight_kg"
})

print("Renamed columns:")
print(df.columns)

Renamed columns:
Index(['harvest_date', 'bunches_24ha', 'bunches_8ha', 'bunches_2ha',
       'total_bunches', 'total_weight_kg', 'avg_bunch_weight_kg'],
      dtype='object')


## Step 3: Clean Invalid Placeholders

Here we replace spreadsheet-style error values, such as `#DIV/0!`, with proper missing values (`NaN`). This makes the dataset safer to convert and validate.


In [200]:
df = df.replace("#DIV/0!", np.nan)

print(df.isna().sum())
display(df[df["avg_bunch_weight_kg"].isna()].head())

harvest_date            0
bunches_24ha           27
bunches_8ha            29
bunches_2ha            29
total_bunches          18
total_weight_kg        26
avg_bunch_weight_kg    44
dtype: int64


,harvest_date,bunches_24ha,bunches_8ha,bunches_2ha,total_bunches,total_weight_kg,avg_bunch_weight_kg
111,8/6/18,1075.0,NaN,NaN,NaN,25510.0,NaN
180,1/14/20,NaN,NaN,NaN,NaN,12646.0,NaN
181,1/21/20,NaN,NaN,NaN,NaN,6650.0,NaN
184,2/13/20,654.0,NaN,NaN,NaN,7036.0,NaN
200,6/11/20,NaN,NaN,NaN,NaN,8210.0,NaN


## Step 4: Convert Data Types

Once invalid placeholders are cleaned, we convert the date column to datetime and the numeric fields to numeric types so calculations and filtering behave correctly.


In [201]:
print("Column data types before conversion:")
print(df.dtypes)

df["harvest_date"] = pd.to_datetime(df["harvest_date"])

numeric_cols = [
    "bunches_24ha",
    "bunches_8ha",
    "bunches_2ha",
    "total_bunches",
    "total_weight_kg",
    "avg_bunch_weight_kg"
]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

print("\nColumn data types after conversion:")
print(df.dtypes)

Column data types before conversion:
harvest_date            object
bunches_24ha           float64
bunches_8ha            float64
bunches_2ha            float64
total_bunches          float64
total_weight_kg        float64
avg_bunch_weight_kg    float64
dtype: object

Column data types after conversion:
harvest_date           datetime64[ns]
bunches_24ha                  float64
bunches_8ha                   float64
bunches_2ha                   float64
total_bunches                 float64
total_weight_kg               float64
avg_bunch_weight_kg           float64
dtype: object


## Step 5: Profile Missing Values

Before filling or dropping anything, we inspect the pattern of missingness. This helps us distinguish between values that are truly unavailable and values that may be derived later from other columns.


In [202]:
# Print summary of missing values by column
print(df.isna().sum().sort_values(ascending=False))

# Display some rows with missing values
display(df[df.isna().any(axis=1)].head(10))

# Display all rows with missing values
# missing_rows = df[df.isna().any(axis=1)].copy()
# display(missing_rows)

avg_bunch_weight_kg    44
bunches_8ha            29
bunches_2ha            29
bunches_24ha           27
total_weight_kg        26
total_bunches          18
harvest_date            0
dtype: int64


,harvest_date,bunches_24ha,bunches_8ha,bunches_2ha,total_bunches,total_weight_kg,avg_bunch_weight_kg
0,2015-12-04,NaN,NaN,NaN,622.0,1320.0,2.122187
1,2015-12-19,NaN,NaN,NaN,570.0,1470.0,2.578947
2,2015-12-30,NaN,NaN,NaN,420.0,1190.0,2.833333
3,2016-01-05,NaN,NaN,NaN,680.0,1790.0,2.632353
4,2016-01-12,NaN,NaN,NaN,735.0,1920.0,2.612245
5,2016-01-20,NaN,NaN,NaN,937.0,2410.0,2.572038
99,2018-05-16,NaN,NaN,NaN,1981.0,16050.0,8.101969
111,2018-08-06,1075.0,NaN,NaN,NaN,25510.0,NaN
126,2018-11-19,NaN,NaN,NaN,648.0,5130.0,7.916667
127,2018-11-28,NaN,NaN,NaN,952.0,7146.0,7.506303


## Step 6: Validate Core Business Rules

We validate the two key relationships in the dataset using rows with the required inputs present:
- `total_bunches = bunches_24ha + bunches_8ha + bunches_2ha`
- `avg_bunch_weight_kg = total_weight_kg / total_bunches`


In [203]:
# Validate that total_bunches = bunches_24ha + bunches_8ha + bunches_2ha

total_check = df.dropna(subset=[
    "bunches_24ha",
    "bunches_8ha",
    "bunches_2ha",
    "total_bunches"
]).copy()

total_check["calculated_total_bunches"] = (
    total_check["bunches_24ha"] +
    total_check["bunches_8ha"] +
    total_check["bunches_2ha"]
)

total_mismatches = total_check[
    total_check["calculated_total_bunches"] != total_check["total_bunches"]
]

print("Rows checked:", len(total_check))
print("Mismatches found:", len(total_mismatches))
display(total_mismatches.head())


Rows checked: 418
Mismatches found: 0


,harvest_date,bunches_24ha,bunches_8ha,bunches_2ha,total_bunches,total_weight_kg,avg_bunch_weight_kg,calculated_total_bunches


In [204]:
# Validate that avg_bunch_weight_kg = total_weight_kg / total_bunches

avg_check = df.dropna(subset=[
    "total_weight_kg",
    "total_bunches",
    "avg_bunch_weight_kg"
]).copy()

avg_check["calculated_avg_bunch_weight_kg"] = (
    avg_check["total_weight_kg"] / avg_check["total_bunches"]
)

avg_mismatches = avg_check[
    avg_check["calculated_avg_bunch_weight_kg"].round(2) !=
    avg_check["avg_bunch_weight_kg"].round(2)
]

print("Rows checked:", len(avg_check))
print("Mismatches found:", len(avg_mismatches))
display(avg_mismatches.head(10))


Rows checked: 403
Mismatches found: 0


,harvest_date,bunches_24ha,bunches_8ha,bunches_2ha,total_bunches,total_weight_kg,avg_bunch_weight_kg,calculated_avg_bunch_weight_kg


## Step 7: Create Validation Flags And Export The Base Dataset

In this final step, we store the results of the checks inside the dataset itself. We also round average bunch weight to two decimals so the validated file uses a practical business precision.


In [205]:
# Flag rows that still contain at least one missing value.
df["has_missing_values"] = df.isna().any(axis=1)

# Recalculate total bunches so we can compare the recorded total with the lot-level sum.
df["calculated_total_bunches"] = (
    df["bunches_24ha"] +
    df["bunches_8ha"] +
    df["bunches_2ha"]
)

# Mark whether the recorded total is valid when all required inputs are present.
df["total_bunches_valid"] = np.where(
    df[["bunches_24ha", "bunches_8ha", "bunches_2ha", "total_bunches"]].notna().all(axis=1),
    df["calculated_total_bunches"] == df["total_bunches"],
    np.nan
)
df["total_bunches_valid"] = df["total_bunches_valid"].astype("boolean")

# Use two decimal places as the project standard for average bunch weight.
df["avg_bunch_weight_kg"] = df["avg_bunch_weight_kg"].round(2)
df["calculated_avg_bunch_weight_kg"] = (
    df["total_weight_kg"] / df["total_bunches"]
).round(2)

# Mark whether the recorded average matches the calculated average when validation is possible.
df["avg_bunch_weight_valid"] = np.where(
    df[["total_weight_kg", "total_bunches", "avg_bunch_weight_kg"]].notna().all(axis=1),
    np.isclose(df["calculated_avg_bunch_weight_kg"], df["avg_bunch_weight_kg"], atol=0.01),
    np.nan
)
df["avg_bunch_weight_valid"] = df["avg_bunch_weight_valid"].astype("boolean")


In [206]:
# Review the cleaned structure and the new validation flags.
display(df.head())
print(df[["has_missing_values", "total_bunches_valid", "avg_bunch_weight_valid"]].head(10))

,harvest_date,bunches_24ha,bunches_8ha,bunches_2ha,total_bunches,total_weight_kg,avg_bunch_weight_kg,has_missing_values,calculated_total_bunches,total_bunches_valid,calculated_avg_bunch_weight_kg,avg_bunch_weight_valid
0,2015-12-04,NaN,NaN,NaN,622.0,1320.0,2.12,True,NaN,<NA>,2.12,True
1,2015-12-19,NaN,NaN,NaN,570.0,1470.0,2.58,True,NaN,<NA>,2.58,True
2,2015-12-30,NaN,NaN,NaN,420.0,1190.0,2.83,True,NaN,<NA>,2.83,True
3,2016-01-05,NaN,NaN,NaN,680.0,1790.0,2.63,True,NaN,<NA>,2.63,True
4,2016-01-12,NaN,NaN,NaN,735.0,1920.0,2.61,True,NaN,<NA>,2.61,True


   has_missing_values  total_bunches_valid  avg_bunch_weight_valid
0                True                 <NA>                    True
1                True                 <NA>                    True
2                True                 <NA>                    True
3                True                 <NA>                    True
4                True                 <NA>                    True
5                True                 <NA>                    True
6               False                 True                    True
7               False                 True                    True
8               False                 True                    True
9               False                 True                    True


In [207]:
# Export the validated base dataset for the next notebook.
df.to_csv("../data/palma_validated.csv", index=False)
print("Validated dataset saved to ../data/palma_validated.csv")


Validated dataset saved to ../data/palma_validated.csv


### Notebook Outcome

This notebook produced a validated base dataset with standardized column names, cleaned placeholder values, corrected data types, rounded average-weight values, and reusable validation flags. The next notebook can focus on missing-value strategy, exploratory analysis, and feature preparation.
